In [ ]:
import numpy as np
import keras
import matplotlib.pyplot as plt
from keras.models import Model
from keras.layers import Input, Dense, Flatten, Conv2D, MaxPooling2D, Dropout
from keras.utils import plot_model
from keras.datasets import fashion_mnist

In [ ]:
# Definition of class labels
labels = ["T-shirt/top","Trouser","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Ankle boot"]

In [ ]:
# Load data
(x_train,y_train), (x_test, y_test) = fashion_mnist.load_data()
# Preview
print("label ",labels[y_train[0]])
plt.imshow(x_train[0], cmap='gray')

In [ ]:
# Data preprocessing
# Convert to float32, models often perform better with numerical data in float form.
# Normalize data to range [0, 1]
x_train = x_train.astype('float32') / 255
x_test = x_test.astype('float32') / 255

In [ ]:
# Convert labels to categories
# One-hot encoding on the training label set
# Convert class labels into binary form as vectors of zeros and ones
# 3 -> [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
num_classes = 10
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

In [ ]:
# Define model using Functional API
# Conv2D(number of filters,...)
input_shape=(28,28,1)
inputs = Input(shape=input_shape)
x = Conv2D(16, kernel_size=(3, 3), padding='same',activation='relu')(inputs)#64
x = Conv2D(16, kernel_size=(3, 3), padding='same',activation='relu')(x)
x = MaxPooling2D(pool_size=(2, 2))(x)
x = Dropout(0.5)(x)
x = Flatten()(x)
x = Dense(128, activation='relu')(x)#128
x = Dropout(0.25)(x)
outputs = Dense(num_classes, activation='softmax')(x)
model = Model(inputs=inputs, outputs=outputs)

In [ ]:
# Compile the model
# 'categorical_crossentropy' average of the logarithm of predicted probabilities for the true class
# Adam Optimizer utilizes adaptive moment estimation to efficiently adjust model weights during training
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# Train the model
# batch_size number of training samples used for one weight update of the model during one training iteration
history = model.fit(x_train, y_train, batch_size=1000, epochs=3, verbose='auto', validation_data=(x_test, y_test))

In [ ]:
import pandas as pd
# Creating a plot with history
df = pd.DataFrame(history.history)
ax = df.plot()
# Save the plot to a file
fig = ax.get_figure()
fig.savefig('history_plot.png')

In [ ]:
# Model evaluation
loss, accuracy = model.evaluate(x_test, y_test, verbose='auto')
print('Test loss:', loss)
print('Test accuracy:', accuracy)

In [ ]:
# Model visualization
model.summary()
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True)

In [ ]:
# Exporting CNN model
model.save("my_model_fashion_mnist.keras")

In [ ]:
# Displaying misclassified examples
predictions = np.argmax(model.predict(x_test), axis=1);
y_test_flat = np.argmax(y_test, axis=1);
incorrect_indices = np.nonzero(predictions != y_test_flat)[0]

for i in range(5):
    idx = incorrect_indices[i]
    print("Misclassified example number", i+1)
    plt.imshow(x_test[idx].reshape(28, 28), cmap='gray')
    plt.xlabel(f"True label:  {labels[y_test_flat[idx]]}, Predicted label:  {labels[predictions[idx]]}")
    plt.show()

In [ ]:
# Visualisation
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# Creating a confusion matrix
cm = confusion_matrix(y_test_flat, predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)

# Creating the plot
fig, ax = plt.subplots(figsize=(10, 10))
disp = disp.plot(xticks_rotation='vertical', ax=ax, cmap='summer')

# Save the plot to a file
plt.savefig('confusion_matrix.png')
plt.show()